# Session 1 — U-Net pretraining, freezing ablation, INR experiments

Self-contained: depends on no other session. The INR runs dominate the
runtime. Expect several hours; every step is resumable, so re-running a
cell after a disconnect skips what already finished.

## 1. Setup

In [ ]:
!git clone -b improve_transformer https://github.com/rifatozkurt/FullWaveformInversion
%cd FullWaveformInversion

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f'{p.name}, {p.total_memory/1e9:.1f} GB')
    # A single adjoint evaluation allocates ~3 GB. Anything under ~8 GB
    # means running one job at a time and nothing else on the GPU.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/fwi_thesis'
!mkdir -p {OUT}

## 2. Data

`extended/` (ids 0-14999) is training data; `eval/` (ids 15000-15999) is
held out. They do NOT overlap. Restore both from Drive if you have them
zipped there, otherwise generate (slow).

In [ ]:
# Restore from Drive (fast path)
!unzip -q -o {OUT}/data/extended.zip -d /content/  || echo 'no extended.zip'
!unzip -q -o {OUT}/data/eval.zip     -d /content/  || echo 'no eval.zip'
!ls /content/extended | head -3 ; ls /content/eval | head -3

# --- alternative: generate instead (hours) ---
# !python scripts/generate_train_data_colab.py \
#     --config configs/config_final.yaml --output-dir /content/extended \
#     --start-case-id 0 --number-of-cases 15000 --case-batch-size 4 --no-overwrite

## 3a. Pretrain the 800-sample U-Net

This is the baseline the freezing ablation fine-tunes. 800 matches
Singh et al., and keeps this session independent of session 2.
Retraining is required regardless: the gamma-MSE definition, the input
normalization and the decoder BatchNorms all changed.

In [ ]:
!python scripts/pretrain.py \
    --config configs/config_final.yaml \
    --data-dir /content/extended \
    --output-dir models/final \
    --run-dir runs/final/unet_pretraining_800

## 3b. Freezing ablation (Experiment 1)

Four modes. `random_encoder` is the control that makes this an
adjudication: a randomly re-initialized, frozen encoder. If it matches
the frozen *pretrained* encoder, what transfers is not the features.

In [ ]:
!python scripts/run_freezing_ablation.py \
    --config configs/config_final.yaml \
    --data-dir /content/eval \
    --run-dir runs/final/freezing \
    --modes encoder,decoder,random_encoder,none

## 3c. Tune the INR learning rates in-distribution (recommended, ~20 min)

The rates currently in the config were selected on `data/casestudy/`,
which holds deliberately unusual out-of-distribution shapes. A partial
in-distribution re-measurement showed the difference is real: IG-FWI
scored 0.408x trivial on the case-study sample but 0.868x on eval case
15002, and a grid rate of 3e-1 — merely mediocre there — **diverged**
in distribution (6.5x). Run this, then paste the printed values into
`configs/config_final.yaml` before the experiments below.

Skip it only if you accept selecting hyperparameters on a different
distribution from the one you report on, and say so in the thesis.

In [ ]:
!python scripts/tune_inr_learning_rates.py \
    --config configs/config_final.yaml \
    --data-dir /content/eval \
    --run-dir runs/final/inr_tuning \
    --epochs 5

## 3d. INR experiments (Experiment 2)

Four ansätze on two held-out cases. Add the `_centered` variants to
`--methods` if you want them; they roughly double the runtime.

In [ ]:
!python scripts/run_all_experiments.py \
    --config configs/config_final.yaml \
    --methods inr_siren_fwi,inr_lr_fwi,inr_mpe_fwi,inr_ig_fwi \
    --cases 15000,15001 \
    --data-dir /content/eval \
    --run-dir runs/final/inr

_Optional: the centred variants._

In [ ]:
# !python scripts/run_all_experiments.py \
#     --config configs/config_final.yaml \
#     --methods inr_siren_centered_fwi,inr_mpe_centered_fwi,inr_ig_centered_fwi \
#     --cases 15000,15001 --data-dir /content/eval \
#     --run-dir runs/final/inr_centered

## 4. Save everything to Drive

`runs/` holds every history, CSV and figure; `models/` holds the
checkpoints. Zip both so a disconnect does not lose the session.

In [ ]:
SESSION = 'session1'
!mkdir -p {OUT}/{SESSION}
!zip -qr /content/runs.zip runs
!cp /content/runs.zip {OUT}/{SESSION}/runs.zip
!zip -qr /content/models.zip models
!cp /content/models.zip {OUT}/{SESSION}/models.zip
print('saved to', OUT + '/' + SESSION)

## 5. Look at the figures before you disconnect

In [ ]:
from IPython.display import Image, display
import pathlib
for p in sorted(pathlib.Path('runs/final').rglob('report/*.png')):
    print(p)
    display(Image(str(p)))